# 22 Research Outputs

目的: full run の結果を、論文・発表で使いやすい図表、表、HTML index、任意の CV overlay mp4 にまとめます。図表や動画内のラベルは英語で出します。

Required inputs:

- `predictions/{FINAL_RUN_ID}/predictions_v1.parquet`
- `predictions/{FINAL_RUN_ID}/metrics_v1.json`
- `predictions/{FINAL_RUN_ID}/fusion_input_audit_v1.parquet`
- `clips/{FULL_RUN_ID}/clips_v1.parquet`
- optional visual evidence: `debug/{FULL_RUN_ID}/frames`, `detections/`, `tracks/`, `pose2d/`, `objects/`, `homography/`

Created outputs:

- `reports/research_outputs/{FINAL_RUN_ID}/index.html`
- `reports/research_outputs/{FINAL_RUN_ID}/summary.json`
- `reports/research_outputs/{FINAL_RUN_ID}/figures/*`
- `reports/research_outputs/{FINAL_RUN_ID}/tables/*`
  - includes `model_design.csv`, `provenance_checks.csv`, `method_evaluation_sample_counts.csv`, `method_baseline_comparison.csv`
- optional: `reports/research_outputs/{FINAL_RUN_ID}/videos/cv_overlays/*.mp4`

この notebook は元の prediction / clip / CV artifact を上書きしません。overlay mp4 を作る場合も、研究出力 bundle 内だけに manifest / progress / video を書きます。既存 report が `clip_rows=0` などを表示していても、この notebook は full run id 側の clip/CV artifact を直接見に行きます。


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return
    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    drive.mount(str(mountpoint))


def _is_repo_dir(path: Path) -> bool:
    return (path / 'src' / 'sport_pipeline' / '__init__.py').exists() and (path / 'configs').exists()


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates = []
    if os.environ.get('BATTING_CODE_ROOT'):
        candidates.append(Path(os.environ['BATTING_CODE_ROOT']))
    candidates.extend([fixed, Path.cwd(), *Path.cwd().parents])
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError('sport_pipeline repo が見つかりません。BATTING_CODE_ROOT か Drive 配置を確認してください。\n' + '\n'.join(checked))


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.pipeline.run_profile import load_run_profile, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
FINAL_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('FINAL_RUN_ID =', FINAL_RUN_ID)

EXPECTED_BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')

def _version_ok(value):
    return '_v2' in str(value or '') and '_v1' not in str(value or '')

def _assert_v2_research_runtime():
    run_ids = RUN_PROFILE.get('run_ids', {})
    namespace = RUN_PROFILE.get('artifact_namespace', {})
    fusion_sources = RUN_PROFILE.get('execution', {}).get('fusion', {}).get('source_runs', [])
    problems = []
    if BASE_DIR != EXPECTED_BASE_DIR:
        problems.append(f'BASE_DIR must be {EXPECTED_BASE_DIR}, got {BASE_DIR}')
    if RUN_PROFILE_NAME != 'mlb_2024_2026_real_colab_v2.json':
        problems.append(f'RUN_PROFILE_NAME must be mlb_2024_2026_real_colab_v2.json, got {RUN_PROFILE_NAME}')
    for key, value in run_ids.items():
        if value and not _version_ok(value):
            problems.append(f'run_ids.{key} is not v2-isolated: {value}')
    for key, value in namespace.items():
        if value and not _version_ok(value):
            problems.append(f'artifact_namespace.{key} is not v2-isolated: {value}')
    for value in fusion_sources:
        if value and not _version_ok(value):
            problems.append(f'fusion source_run is not v2-isolated: {value}')
    if run_ids.get('vlm_run_id') not in fusion_sources:
        problems.append(f'fusion source_runs missing VLM run: {run_ids.get("vlm_run_id")}')
    if f'{run_ids.get("vlm_run_id")}_player_season_projection' not in fusion_sources:
        problems.append(f'fusion source_runs missing VLM player-season projection: {run_ids.get("vlm_run_id")}_player_season_projection')
    if problems:
        raise RuntimeError('22 v2 research runtime contract failed:\n- ' + '\n- '.join(problems))
    print('22 v2 research runtime contract = ok')

_assert_v2_research_runtime()


In [ ]:
from sport_pipeline.reports.run_selection import report_run_candidates, prediction_artifact_path, metrics_artifact_path

print('主要 prediction runs:')
for rid in report_run_candidates(RUN_PROFILE, include_smoke=False):
    pred = prediction_artifact_path(BASE_DIR, rid)
    metrics = metrics_artifact_path(BASE_DIR, rid)
    print(f'- {rid}')
    print('  predictions:', pred, pred.exists())
    print('  metrics    :', metrics, metrics.exists())

print('\nvisual / mechanics artifacts:')
visual_paths = {
    'clips': BASE_DIR / 'clips' / FULL_RUN_ID / 'clips_v1.parquet',
    'candidate_segments': BASE_DIR / 'clips' / FULL_RUN_ID / 'candidate_segments_v1.parquet',
    'contact_frames': BASE_DIR / 'debug' / FULL_RUN_ID / 'frames',
    'detections': BASE_DIR / 'detections' / FULL_RUN_ID / 'detections_v1.parquet',
    'tracks': BASE_DIR / 'tracks' / FULL_RUN_ID / 'tracks_v1.parquet',
    'pose2d': BASE_DIR / 'pose2d' / FULL_RUN_ID / 'pose2d_v1.parquet',
    'bat_detection': BASE_DIR / 'objects' / FULL_RUN_ID / 'bat_detection_v1.parquet',
    'bat_line': BASE_DIR / 'objects' / FULL_RUN_ID / 'bat_line_v1.parquet',
    'homography': BASE_DIR / 'homography' / FULL_RUN_ID / 'homography_v1.parquet',
    'existing_overlay_manifest': BASE_DIR / 'debug' / FULL_RUN_ID / 'cv_overlay_videos_v1.parquet',
}
for name, path in visual_paths.items():
    if path.is_dir():
        count = len(list(path.glob('*')))
        print(f'- {name}: {path} exists={path.exists()} files={count}')
    else:
        print(f'- {name}: {path} exists={path.exists()}')

required = [prediction_artifact_path(BASE_DIR, FINAL_RUN_ID), metrics_artifact_path(BASE_DIR, FINAL_RUN_ID)]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('final fusion artifact がありません。先に 15_full_fusion.ipynb を確認してください: ' + '\n'.join(missing))


In [ ]:
MAKE_OVERLAY_VIDEOS = True
MAX_OVERLAY_CLIPS = 12
MAX_PREDICTION_PLOT_ROWS = 60000

print('MAKE_OVERLAY_VIDEOS =', MAKE_OVERLAY_VIDEOS)
print('MAX_OVERLAY_CLIPS =', MAX_OVERLAY_CLIPS)
print('MAX_PREDICTION_PLOT_ROWS =', MAX_PREDICTION_PLOT_ROWS)
print('overlay は研究出力 bundle 内だけに作ります。時間を短くしたい場合は MAKE_OVERLAY_VIDEOS=False にしてください。')


In [ ]:
from sport_pipeline.reports.research_outputs import build_research_outputs

summary = build_research_outputs(
    base_dir=BASE_DIR,
    run_profile=RUN_PROFILE,
    final_run_id=FINAL_RUN_ID,
    full_run_id=FULL_RUN_ID,
    make_overlay_videos=MAKE_OVERLAY_VIDEOS,
    max_overlay_clips=MAX_OVERLAY_CLIPS,
    max_prediction_plot_rows=MAX_PREDICTION_PLOT_ROWS,
)

print('index_html =', summary['outputs']['index_html'])
print('summary_json =', summary['outputs']['summary_json'])
print('figures =', summary['outputs']['figures'])
print('tables =', summary['outputs']['tables'])
print('videos =', summary['outputs']['videos'])
print('metrics_rows =', summary['metrics_rows'])
print('overlay_summary =', summary['overlay_summary'])
method_comparison = summary.get('method_comparison', {})
print('method_comparison =', {
    'method_metrics_rows': method_comparison.get('method_metrics_rows'),
    'sample_count_rows': method_comparison.get('sample_count_rows'),
    'same_sample_metric_rows': method_comparison.get('same_sample_metric_rows'),
    'baseline_comparison_rows': method_comparison.get('baseline_comparison_rows'),
    'figures': method_comparison.get('figures'),
})
provenance = summary.get('provenance_checks', {})
print('provenance_status =', provenance.get('overall_status'))
for check in provenance.get('checks', []):
    if check.get('status') != 'ok':
        print('PROVENANCE WARNING:', check)


In [ ]:
from IPython.display import HTML, IFrame, Image, SVG, Video, display

index_html = Path(summary['outputs']['index_html'])
figures_dir = Path(summary['outputs']['figures'])
videos_dir = Path(summary['outputs']['videos'])

print('研究出力 HTML:', index_html)
display(IFrame(src=str(index_html), width='100%', height=900))

method_svg = figures_dir / 'method_overview.svg'
if method_svg.exists():
    display(SVG(filename=str(method_svg)))

for figure_name in [
    'model_metrics_mae.png',
    'target_availability.png',
    'fusion_input_provenance.png',
    'method_available_rows_by_target.png',
    'method_primary_metric_matrix.png',
    'method_same_sample_metric_matrix.png',
    'method_delta_vs_context.png',
    'method_delta_vs_fusion.png',
    'prediction_scatter_ev_la.png',
    'residual_distribution_ev_la.png',
    'contact_frame_sheet.jpg',
]:
    figure = figures_dir / figure_name
    if figure.exists():
        print('\nFIGURE:', figure)
        display(Image(filename=str(figure)))

overlay_videos = sorted(videos_dir.glob('**/*.mp4'))
print('\noverlay videos =', len(overlay_videos))
if overlay_videos:
    print('sample overlay:', overlay_videos[0])
    display(Video(str(overlay_videos[0]), embed=False, width=720))
else:
    print('overlay mp4 がありません。pose/detection artifact が存在するか、OpenCV が入っているかを確認してください。')

print('\nNEXT: 09_report_builder.ipynb を再実行して通常 report を更新し、その後この 22 を再実行すると、clip_rows=0 の古い表示と研究出力を分けて確認できます。')
